# Through-Sample Interface Study

Stage D notebook. The input is an air-side `surface_plane` field. The sample branch uses a planar interface, explicit in-medium coordinates, and labels the correction branch as `ideal_numerical_correction` unless a hardware route implements it.

In [1]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_regime, vbb_sample_study
from vbb_study.publication import lab_realism as lab_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_fig = PATHS["figures"] / "stage_d"
out_csv = PATHS["csv"] / "stage_d"
out_fig.mkdir(parents=True, exist_ok=True)
out_csv.mkdir(parents=True, exist_ok=True)

base0 = bt.default_config(PRESET)
notebook_grid = replace(base0.grid, axial_points=25, coarse_scan_points=17, crop_pixels=160)
base0 = replace(base0, grid=notebook_grid)

def _route_method(method):
    return "physical_axicon" if method == "physical" else "holographic_axicon"

def _route_hardware_status(method, variant):
    if variant == "ideal":
        return "simulation_only"
    return "future_hardware_required" if method == "physical" else "current_lab_realizable"


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(stage='lab_realism')
try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [2]:
def configured_case(method, regime, variant):
    cfg = replace(base0, generation_method=method)
    cfg = vbb_regime.config_for_regime(cfg, regime)
    if method == "physical":
        levels = 256 if variant == "lab" else None
        cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=levels, slm2_conjugate_mode="full"))
        path = "ideal"
    else:
        path = "realistic" if variant == "lab" else "ideal"
    return cfg, path

def row_for(label, regime, method, variant, beam, uncorrected, corrected):
    cm = corrected.metrics
    um = uncorrected.metrics
    return {
        "case_id": label,
        "regime": regime,
        "route_generation_method": _route_method(method),
        "method": method,
        "variant": variant,
        "path": "realistic" if method == "holographic" and variant == "lab" else "ideal",
        "beam_air_zone_um": beam["metrics"]["canonical_zone_um"],
        "sample_uncorrected_zone_um": um["canonical_zone_um"],
        "sample_corrected_zone_um": cm["canonical_zone_um"],
        "strict_bessel_region_um": cm["strict_bessel_region_um"],
        "sample_corrected_peak_z_um": cm["peak_z_um"],
        "ring_or_core_um": cm["ring_radius_um"] if int(cm["ell"]) else cm["core_radius_um"],
        "peak_fluence_J_cm2": cm["peak_fluence_J_cm2"],
        "side_to_core_peak_ratio": cm["side_to_core_peak_ratio"],
        "medium_n": cm["medium_n"],
        "surface_transmission": cm["surface_transmission"],
        "interface_correction_label_uncorrected": um["interface_correction_label"],
        "interface_correction_label_corrected": cm["interface_correction_label"],
        "interface_correction_implementation": cm["interface_correction_implementation"],
        "corrected_rel_l2_to_no_interface": cm["corrected_rel_l2_to_no_interface"],
        "uncorrected_rel_l2_to_no_interface": cm["uncorrected_rel_l2_to_no_interface"],
        "phase_only_power_relative_error": cm["phase_only_power_relative_error"],
        "spherical_after_waves": cm["interface_spherical_after_waves"],
    }


In [3]:
rows = []
correction_rows = []
results = {}
for regime in ("general", "limits"):
    for method in ("holographic", "physical"):
        for variant in ("ideal", "lab"):
            cfg, path = configured_case(method, regime, variant)
            label = f"{regime}_{method}_{variant}"
            beam = bt.run_case(cfg, preset=PRESET, path=path, case_id=f"{label}_beam")
            uncorrected = vbb_sample_study.run_through_sample(beam["surface_field"], cfg, correct_interface=False)
            corrected = vbb_sample_study.run_through_sample(beam["surface_field"], cfg, correct_interface=True)
            rows.append(row_for(label, regime, method, variant, beam, uncorrected, corrected))
            for sample in (uncorrected, corrected):
                sm = sample.metrics
                correction_rows.append({
                    "case_id": f"{label}_{sm['interface_correction_label']}",
                    "regime": regime,
                    "method": method,
                    "variant": variant,
                    "path": path,
                    "route_generation_method": _route_method(method),
                    "canonical_zone_um": sm["canonical_zone_um"],
                    "strict_bessel_region_um": sm["strict_bessel_region_um"],
                    "interface_correction_label": sm["interface_correction_label"],
                    "interface_correction_implementation": sm["interface_correction_implementation"],
                    "corrected_rel_l2_to_no_interface": sm["corrected_rel_l2_to_no_interface"],
                    "uncorrected_rel_l2_to_no_interface": sm["uncorrected_rel_l2_to_no_interface"],
                    "phase_only_power_relative_error": sm["phase_only_power_relative_error"],
                    "medium_n": sm["medium_n"],
                    "surface_transmission": sm["surface_transmission"],
                })
            results[label] = {"beam": beam, "uncorrected": uncorrected, "corrected": corrected}

summary = pd.DataFrame(rows)
summary = lab_schema.with_lab_realism_metadata(
    summary,
    generation_method="interface_corrected_numerical",
    model_level="interface_model",
    hardware_status="diagnostic_only",
    plane_label="in_medium_plane",
    coordinate_frame="sample_plane_um_z_from_surface_in_medium",
    run_id=RUN_ID,
    preset=PRESET,
    path="through_sample",
)
summary.to_csv(out_csv / "through_sample_summary.csv", index=False)

correction_rows_stamped = []
for row in correction_rows:
    label = row["interface_correction_label"]
    lab_schema.annotate_lab_realism_row(
        row,
        generation_method="interface_corrected_numerical" if label == "ideal_numerical_correction" else "interface_uncorrected",
        model_level="interface_model",
        hardware_status="diagnostic_only" if label == "ideal_numerical_correction" else _route_hardware_status(row["method"], row["variant"]),
        plane_label="in_medium_plane",
        coordinate_frame="sample_plane_um_z_from_surface_in_medium",
        run_id=RUN_ID,
        preset=PRESET,
        path=row["path"],
    )
    correction_rows_stamped.append(row)
interface_correction_summary = lab_schema.ordered_lab_realism_frame(correction_rows_stamped)
interface_correction_summary.to_csv(out_csv / "interface_correction_summary.csv", index=False)
corrected_vs_uncorrected_metrics = interface_correction_summary.copy()
corrected_vs_uncorrected_metrics.to_csv(out_csv / "corrected_vs_uncorrected_metrics.csv", index=False)
summary


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,generation_method,model_level,hardware_status,plane_label,...,side_to_core_peak_ratio,medium_n,surface_transmission,interface_correction_label_uncorrected,interface_correction_label_corrected,interface_correction_implementation,corrected_rel_l2_to_no_interface,uncorrected_rel_l2_to_no_interface,phase_only_power_relative_error,spherical_after_waves
0,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,general_holographic_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.429350,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007046,1.350925,1.773750e-16,-0.000089
1,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,general_holographic_lab,fast,realistic,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.331575,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007087,1.357514,1.592704e-16,-0.000161
2,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,general_physical_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.592451,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.006995,1.370763,1.865593e-16,-0.000089
3,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,general_physical_lab,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.592956,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.006995,1.370788,0.000000e+00,-0.000089
4,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,limits_holographic_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.500058,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007084,1.424083,0.000000e+00,-0.000089
5,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,limits_holographic_lab,fast,realistic,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,1.000000,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007907,1.101286,0.000000e+00,-0.000161
6,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,limits_physical_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.155709,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007092,1.426270,2.081042e-16,-0.000089
7,20260603T205637Z,2026-06-03T20:59:52.859324+00:00,1.0.0,limits_physical_lab,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.155720,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007092,1.426208,0.000000e+00,-0.000089


In [4]:
for label in ("general_holographic_lab", "general_physical_lab"):
    vbb_sample_study.plot_sample_result_comparison(
        results[label]["uncorrected"],
        results[label]["corrected"],
        out_fig / f"through_sample_{label}.png",
        title=f"Through-sample {label.replace('_', ' ')}",
    )
